# Alignment Transfer for S2UT
GPUランタイムを選択してください。リポジトリURLとコードrevisionは設定済みです。モデル・前処理ロジックはCLIにあります。全セルは同じ設定で再実行でき、完成済みartifactとcheckpointから自動再開します。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git' #@param {type:"string"}
REPO_REVISION = 'ede613269b1740b52e8e72a2ee7a4b39f5a1ed27' #@param {type:"string"}
ROOT = '/content/drive/MyDrive/alignment-transfer-s2ut-data/experiment-002' #@param {type:"string"}
CORPUS = '/content/drive/MyDrive/ja-en-direct-s2st-corpus-data' #@param {type:"string"}
RECIPE = 'controlled' #@param ['controlled', 'strong']
RUNS = ROOT + '/' + RECIPE
ANNEAL_STEPS = 10000 #@param {type:"integer"}
UPDATES = 2 #@param {type:"integer"}
PYTHON = '/content/alignment-venv/bin/python'
os.environ['HF_HOME'] = '/content/huggingface'


In [ ]:
# Clone/install only. Never point outputs at the corpus or benchmark repositories.
import subprocess
from pathlib import Path
assert len(REPO_REVISION) == 40 and 'SET_' not in REPO_URL
assert not Path(ROOT).resolve().is_relative_to(Path(CORPUS).resolve())
if not Path('/content/alignment-transfer-s2ut/.git').exists():
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, '/content/alignment-transfer-s2ut'], check=True)
subprocess.run(['git', '-C', '/content/alignment-transfer-s2ut', 'checkout', '--detach', REPO_REVISION], check=True)
%cd /content/alignment-transfer-s2ut
!python -m pip -q install uv
!uv venv --python 3.10.22 --seed --allow-existing /content/alignment-venv
!{PYTHON} -m pip -q install pip==23.3.2 'setuptools<81' wheel
!{PYTHON} -m pip -q install torch==2.7.1 torchaudio==2.7.1 numpy==1.26.4 Cython==3.2.9
!{PYTHON} -m pip install --no-build-isolation -r requirements-fairseq.txt
!{PYTHON} -m pip install -r requirements.txt
!{PYTHON} -m pytest -q


In [ ]:
# First import only. On resume, retain the existing canonical manifest unchanged.
!{PYTHON} -m alignment_transfer_s2ut.prepare_manifest --input {CORPUS}/production/manifests/releases/accepted.jsonl --corpus-root {CORPUS} --output {ROOT}/manifest.jsonl --resume
!{PYTHON} -m alignment_transfer_s2ut.compare_architectures --t2tt-config configs/t2tt/fisher.yaml --s2ut-config configs/s2ut/fisher.yaml --output {ROOT}/architecture.json --resume


In [ ]:
# Keep the same recipe; resume automatically if a checkpoint exists.
!{PYTHON} -m alignment_transfer_s2ut.train_t2tt --manifest {ROOT}/manifest.jsonl --config configs/t2tt/fisher.yaml --output {ROOT}/t2tt --max-updates {UPDATES} --device cuda --resume-if-exists


In [ ]:
!{PYTHON} -m alignment_transfer_s2ut.extract_t2tt_attention --manifest {ROOT}/manifest.jsonl --checkpoint {ROOT}/t2tt/checkpoint_best.pt --output {ROOT}/mt --device cuda --resume
!{PYTHON} -m alignment_transfer_s2ut.align_speech_text --manifest {ROOT}/manifest.jsonl --config configs/alignment/ctc.yaml --output {ROOT}/ctc --device cuda --resume
!{PYTHON} -m alignment_transfer_s2ut.fetch_assets --config configs/units/hubert_km100.yaml
!{PYTHON} -m alignment_transfer_s2ut.extract_units --manifest {ROOT}/manifest.jsonl --config configs/units/hubert_km100.yaml --output {ROOT}/units --device cuda --resume
!{PYTHON} -m alignment_transfer_s2ut.build_teacher_alignment --manifest {ROOT}/manifest.jsonl --mt-attention {ROOT}/mt --alignment {ROOT}/ctc --units {ROOT}/units --config configs/s2ut/fisher.yaml --output {ROOT}/teacher --resume


In [ ]:
# Audit coverage/uncertainty before training; inspect heatmaps against original audio.
!{PYTHON} -m alignment_transfer_s2ut.diagnose_alignment --manifest {ROOT}/manifest.jsonl --teacher {ROOT}/teacher --units {ROOT}/units --config configs/s2ut/fisher.yaml --split dev --max-samples 500 --heatmaps 20 --source-mapping frontend --prior-smoothing 0.05 --output {ROOT}/alignment-audit --overwrite


`RECIPE`でcontrolled / strongを選択します。strongは文字補助decoder・decoder CTC・CMVN・SpecAugmentを使用し、両recipeとも単調prior対照を含めて比較します。出力先はrecipe別です。UPDATES=2は接続確認、pilotは5000〜20000更新、本学習は全条件共通の更新数を設定してください。過去の実装からのresumeはcontractの不一致で停止するため、新しいROOTを使用します。

In [ ]:
# One recipe, timing and smoothing across all five conditions.
!{PYTHON} -m alignment_transfer_s2ut.train_s2ut --manifest {ROOT}/manifest.jsonl --config configs/s2ut/fisher.yaml --units {ROOT}/units --method baseline --output {RUNS}/baseline --recipe {RECIPE} --prior-smoothing 0.05 --source-mapping frontend --anneal-steps {ANNEAL_STEPS} --validation-interval 1000 --save-interval-updates 1000 --max-updates {UPDATES} --device cuda --resume-if-exists
!{PYTHON} -m alignment_transfer_s2ut.train_s2ut --manifest {ROOT}/manifest.jsonl --config configs/s2ut/fisher.yaml --units {ROOT}/units --teacher {ROOT}/teacher --method distillation --output {RUNS}/distillation --recipe {RECIPE} --prior-smoothing 0.05 --source-mapping frontend --anneal-steps {ANNEAL_STEPS} --validation-interval 1000 --save-interval-updates 1000 --max-updates {UPDATES} --device cuda --resume-if-exists
!{PYTHON} -m alignment_transfer_s2ut.train_s2ut --manifest {ROOT}/manifest.jsonl --config configs/s2ut/fisher.yaml --units {ROOT}/units --teacher {ROOT}/teacher --method guidance --output {RUNS}/guidance --recipe {RECIPE} --prior-smoothing 0.05 --source-mapping frontend --anneal-steps {ANNEAL_STEPS} --validation-interval 1000 --save-interval-updates 1000 --max-updates {UPDATES} --device cuda --resume-if-exists
!{PYTHON} -m alignment_transfer_s2ut.train_s2ut --manifest {ROOT}/manifest.jsonl --config configs/s2ut/fisher.yaml --units {ROOT}/units --teacher {ROOT}/teacher --method annealed_guidance --output {RUNS}/annealed_guidance --recipe {RECIPE} --prior-smoothing 0.05 --source-mapping frontend --anneal-steps {ANNEAL_STEPS} --validation-interval 1000 --save-interval-updates 1000 --max-updates {UPDATES} --device cuda --resume-if-exists
!{PYTHON} -m alignment_transfer_s2ut.train_s2ut --manifest {ROOT}/manifest.jsonl --config configs/s2ut/fisher.yaml --units {ROOT}/units --recipe {RECIPE} --method annealed_guidance --prior-type monotonic --prior-smoothing 0.05 --source-mapping frontend --anneal-steps {ANNEAL_STEPS} --validation-interval 1000 --save-interval-updates 1000 --output {RUNS}/annealed_monotonic --max-updates {UPDATES} --device cuda --resume-if-exists
!{PYTHON} -m alignment_transfer_s2ut.compare_experiments {RUNS}/baseline {RUNS}/distillation {RUNS}/guidance {RUNS}/annealed_guidance {RUNS}/annealed_monotonic


推論入力には`pair_id, ja_audio`だけを持つJSONLを用意します。英語referenceやtranscriptを含む学習manifestは渡しません。評価に使うsplitとID集合を一致させてください。

In [ ]:
SPEECH_ONLY = ROOT + '/test_speech_only.jsonl' #@param {type:"string"}
METHOD = 'annealed_guidance' #@param ['baseline', 'distillation', 'guidance', 'annealed_guidance', 'annealed_monotonic']
!{PYTHON} -m alignment_transfer_s2ut.prepare_inference_input --manifest {ROOT}/manifest.jsonl --split test --output {SPEECH_ONLY} --resume
!{PYTHON} -m alignment_transfer_s2ut.fetch_assets --config configs/s2ut/vocoder.yaml
!{PYTHON} -m alignment_transfer_s2ut.infer --input {SPEECH_ONLY} --checkpoint {RUNS}/{METHOD}/checkpoint_best.pt --vocoder-config configs/s2ut/vocoder.yaml --output {RUNS}/{METHOD}/test-predictions.jsonl --device cuda --resume
!{PYTHON} -m alignment_transfer_s2ut.evaluate --manifest {ROOT}/manifest.jsonl --split test --predictions {RUNS}/{METHOD}/test-predictions.jsonl --units {ROOT}/units --asr-config configs/experiments/evaluation.yaml --output {RUNS}/{METHOD}/test-metrics.json --device cuda --resume
